<a href="https://colab.research.google.com/github/subhanreddy2007-star/FMML-Projects-and-Labs/blob/main/Module3_Lab3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## KNN for Text Classification

This notebook demonstrates how to perform text classification using the K-Nearest Neighbors (KNN) algorithm. We will cover data preparation, text preprocessing, vectorization, model training, and evaluation.

In [36]:
pip install nltk scikit-learn pandas

In [28]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

### 1. Prepare Sample Data

In [29]:
data = {
    'text': [
        'This is a great movie, I loved it!',
        'The film was terrible, what a waste of time.',
        'An amazing cinematic experience, highly recommend.',
        'I hated the plot and the acting was subpar.',
        'Best book I have read in years, truly captivating.',
        'The novel was boring and hard to finish.',
        'Fantastic story, a must-read for everyone.',
        'Worst story ever, completely disappointed.'
    ],
    'category': [
        'positive_movie',
        'negative_movie',
        'positive_movie',
        'negative_movie',
        'positive_book',
        'negative_book',
        'positive_book',
        'negative_book'
    ]
}
df = pd.DataFrame(data)
display(df.head())

,text,category
0,"This is a great movie, I loved it!",positive_movie
1,"The film was terrible, what a waste of time.",negative_movie
2,"An amazing cinematic experience, highly recomm...",positive_movie
3,I hated the plot and the acting was subpar.,negative_movie
4,"Best book I have read in years, truly captivat...",positive_book


### Using a Public Dataset (20 Newsgroups)

In [30]:
from sklearn.datasets import fetch_20newsgroups

newsgroups_data = fetch_20newsgroups(subset='all', remove=('headers', 'footers', 'quotes'))

df_public = pd.DataFrame({'text': newsgroups_data.data, 'category': [newsgroups_data.target_names[i] for i in newsgroups_data.target]})

print(f"Number of samples: {len(df_public)}")
print(f"Number of categories: {len(df_public['category'].unique())}")
display(df_public.head())

Number of samples: 18846
Number of categories: 20


,text,category
0,\n\nI am sure some bashers of Pens fans are pr...,rec.sport.hockey
1,My brother is in the market for a high-perform...,comp.sys.ibm.pc.hardware
2,\n\n\n\n\tFinally you said what you dream abou...,talk.politics.mideast
3,\nThink!\n\nIt's the SCSI card doing the DMA t...,comp.sys.ibm.pc.hardware
4,1) I have an old Jasmine drive which I cann...,comp.sys.mac.hardware


### 2. Text Preprocessing

We'll perform the following preprocessing steps:
- Tokenization
- Lowercasing
- Removing stop words

In [31]:
import nltk
nltk.download('punkt_tab', quiet=True)

stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    if not isinstance(text, str):
        return ''
    tokens = word_tokenize(text)
    filtered_tokens = [word.lower() for word in tokens if word.isalpha() and word.lower() not in stop_words]
    return ' '.join(filtered_tokens)

df_public['processed_text'] = df_public['text'].apply(preprocess_text)
display(df_public.head())

,text,category,processed_text
0,\n\nI am sure some bashers of Pens fans are pr...,rec.sport.hockey,sure bashers pens fans pretty confused lack ki...
1,My brother is in the market for a high-perform...,comp.sys.ibm.pc.hardware,brother market video card supports vesa local ...
2,\n\n\n\n\tFinally you said what you dream abou...,talk.politics.mideast,finally said dream mediterranean new area grea...
3,\nThink!\n\nIt's the SCSI card doing the DMA t...,comp.sys.ibm.pc.hardware,think scsi card dma transfers disks scsi card ...
4,1) I have an old Jasmine drive which I cann...,comp.sys.mac.hardware,old jasmine drive use new system understanding...


### 3. Text Vectorization (TF-IDF)

We will use TF-IDF (Term Frequency-Inverse Document Frequency) to convert our text data into numerical feature vectors. This technique reflects how important a word is to a document in a corpus.

In [32]:
vectorizer = TfidfVectorizer(max_features=10000)
X = vectorizer.fit_transform(df_public['processed_text'])
y = df_public['category']

print("Shape of TF-IDF matrix:", X.shape)

Shape of TF-IDF matrix: (18846, 10000)


### 4. Split Data into Training and Testing Sets

We'll split the dataset into training and testing sets to evaluate the model's performance on unseen data.

In [33]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

Training data shape: (15076, 10000)
Testing data shape: (3770, 10000)


### 5. Train the KNN Classifier

I'll initialize and train a K-Nearest Neighbors classifier, setting `n_neighbors` to 5. This means the classifier will consider the 5 closest neighbors to classify a new data point.

In [34]:
knn_classifier = KNeighborsClassifier(n_neighbors=5)
knn_classifier.fit(X_train, y_train)

print("KNN model trained successfully.")

KNN model trained successfully.


### 6. Evaluate the Model

Finally, we'll evaluate the performance of our trained KNN model on the test set using various classification metrics.

In [35]:
y_pred = knn_classifier.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred, zero_division=0))

Accuracy: 0.12360742705570292

Classification Report:
                           precision    recall  f1-score   support

             alt.atheism       0.09      0.23      0.13       151
           comp.graphics       0.09      0.18      0.12       202
 comp.os.ms-windows.misc       0.09      0.24      0.13       195
comp.sys.ibm.pc.hardware       0.09      0.13      0.11       183
   comp.sys.mac.hardware       0.10      0.14      0.12       205
          comp.windows.x       0.31      0.15      0.20       215
            misc.forsale       0.16      0.10      0.13       193
               rec.autos       0.07      0.22      0.11       196
         rec.motorcycles       0.14      0.10      0.11       168
      rec.sport.baseball       0.10      0.09      0.10       211
        rec.sport.hockey       0.31      0.14      0.19       198
               sci.crypt       0.13      0.08      0.10       201
         sci.electronics       0.18      0.09      0.12       202
                 sci